# Train YOLO26 on Colab — crop-aligned subsets (`below_1000_crop`, `classification_crop`)

**Pick the best GPU first:** *Runtime → Change runtime type → GPU → **A100*** (Colab Pro+); else L4 > V100 > T4.
Then **Runtime → Run all**.

⚠️ **Batch matters.** `yolo26x @1280` is huge — Ultralytics `batch=-1` autobatch collapses to **batch 1**
(~29% of an A100) and crawls. This notebook pins a model-aware batch so the GPU is actually filled.
For ~1.4k training images, **`yolo26l` fills the A100 better and tends to generalize better than `yolo26x`**
(less overfit) — switch `MODEL` in cell 5 to compare.

In [ ]:
# @title 1) Reduce CUDA fragmentation (must run BEFORE torch imports) + GPU check
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import subprocess, torch
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,driver_version","--format=csv,noheader"],
                     capture_output=True, text=True).stdout or "no nvidia-smi")
assert torch.cuda.is_available(), "No CUDA GPU! Runtime -> Change runtime type -> GPU (A100/L4)."
NAME = torch.cuda.get_device_name(0); VRAM = torch.cuda.get_device_properties(0).total_memory/1e9
print(f"GPU: {NAME} | VRAM: {VRAM:.0f} GB | torch {torch.__version__} CUDA {torch.version.cuda}")

In [ ]:
# @title 2) Clone repo + install deps
REPO="/content/repo"
!git clone https://github.com/arupa444/trainDronisight.git $REPO 2>/dev/null || (cd $REPO && git pull)
%cd $REPO
!pip -q install uv && uv pip install --system -e .
import torch; print("CUDA available:", torch.cuda.is_available())

In [ ]:
# @title 3) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# @title 4) Load datasets from Drive -> /content/data (fast local SSD)
import os, glob, zipfile, shutil
DRIVE_DATA = "/content/drive/MyDrive/dronisight"     # <-- adjust if needed
SUBSETS    = ["component_below_1000_crop", "component_classification_crop"]
DEST       = "/content/data/yolo_train_db"; os.makedirs(DEST, exist_ok=True)

def locate(sub):
    for c in [f"{DRIVE_DATA}/yolo_train_db/{sub}", f"{DRIVE_DATA}/{sub}",
              f"{DRIVE_DATA}/yolo_train_db/{sub}.zip", f"{DRIVE_DATA}/{sub}.zip"]:
        if os.path.exists(c): return c
    hits = glob.glob(f"{DRIVE_DATA}/**/{sub}*", recursive=True); return hits[0] if hits else None

for sub in SUBSETS:
    dst = f"{DEST}/{sub}"
    if os.path.isdir(dst) and glob.glob(f"{dst}/images/**/*.jpg", recursive=True):
        print("already present:", sub); continue
    src = locate(sub); assert src, f"could not find {sub} under {DRIVE_DATA}"
    if src.endswith(".zip"):
        print("unzip", src); 
        with zipfile.ZipFile(src) as z: z.extractall(DEST)
        if not os.path.isdir(dst):
            inner = glob.glob(f"{DEST}/**/{sub}", recursive=True)
            if inner: shutil.move(inner[0], dst)
    else:
        print("copy", src); shutil.copytree(src, dst, dirs_exist_ok=True)

os.environ["DRONISIGHT_DATA"] = "/content/data"
import importlib, shared.config as C; importlib.reload(C)
for sub in SUBSETS:
    p = C.YOLO_DB/sub; n = len(glob.glob(f"{p}/images/train/clahe/*.jpg"))
    print(f"{sub}: exists={p.is_dir()} train(clahe)={n}")
    for c in glob.glob(f"{p}/**/*.cache", recursive=True): os.remove(c)

In [ ]:
# @title 5) Train  (model-aware batch fills the GPU; AMP on by default)
from train_yolo.train_components import run

MODEL = "yolo26x.pt"     # <- swap to "yolo26l.pt" for ~5-8x faster + better fit on this data size
IMGSZ = 1280
EPOCHS = {"component_below_1000_crop": 200, "component_classification_crop": 150}

# fixed batch tuned for imgsz 1280 (yolo26x is memory-bound; 26l/26m fit much larger batches).
# autobatch (batch=-1) collapses to 1 on yolo26x@1280, so we pin it.
def pick_batch(model, vram_gb):
    heavy = "26x" in model
    if heavy:    table = [(78, 6), (40, 3), (22, 2), (0, 1)]   # A100-80, A100-40, L4/V100, smaller
    else:        table = [(78, 32), (40, 16), (22, 8), (0, 4)] # 26l / 26m
    for thr, b in table:
        if vram_gb >= thr: return b
    return 1
BATCH = pick_batch(MODEL, VRAM)
print(f"MODEL={MODEL} VRAM={VRAM:.0f}GB -> batch={BATCH} (override BATCH=... to push higher)")

for sub in SUBSETS:
    print("="*70, f"\nTRAIN {sub} epochs={EPOCHS.get(sub,150)} imgsz={IMGSZ} batch={BATCH} model={MODEL}\n","="*70)
    run(subset=sub, version="clahe", epochs=EPOCHS.get(sub,150), imgsz=IMGSZ, batch=BATCH, model=MODEL)

In [ ]:
# @title 6) Save runs/ to Drive (Colab is ephemeral) + show weight paths
from notebooks.colab_utils import save_runs_to_drive
print("saved to:", save_runs_to_drive())
import glob
for w in glob.glob("runs/**/weights/best.pt", recursive=True): print("WEIGHTS:", w)

In [ ]:
# @title 7) (optional) best val mAP@.5 per run
import glob, pandas as pd
for csv in sorted(glob.glob("runs/**/results.csv", recursive=True)):
    df = pd.read_csv(csv); df.columns=[c.strip() for c in df.columns]
    col = next((c for c in df.columns if "mAP50(B)" in c), None)
    if col is not None: print(f"{csv:60s} best mAP@.5={df[col].max():.4f} (epochs {len(df)})")